In [11]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils import spectral_norm
from torchvision import datasets, transforms

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [3]:
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

Done!


In [4]:
transformation = transforms.Compose([transforms.ToTensor()])

In [5]:
train_dataset = datasets.MNIST(
    root=project_root / "data",
    train=True,
    transform=transformation,
    download=True
)

In [6]:
test_dataset = datasets.MNIST(
    root=project_root / "data",
    train=False,
    transform=transformation,
    download=True
)

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [8]:
class StructureEncoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2),  # 14x14
            nn.Conv2d(32, 64, 4, 2, 1), nn.LeakyReLU(0.2), # 7x7
            nn.Flatten()
        )
        self.mu_head = nn.Linear(64 * 7 * 7, latent_dim)
        self.logvar_head = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        features = self.conv(x)
        return self.mu_head(features), self.logvar_head(features)

class BaseDecoder(nn.Module):
    def __init__(self, latent_dim=16, base_channels=64):
        super().__init__()
        self.base_channels = base_channels
        self.fc = nn.Linear(latent_dim, base_channels * 7 * 7)

    def forward(self, z_s):
        blueprint = self.fc(z_s)
        return blueprint.view(-1, self.base_channels, 7, 7)

In [9]:
class AdaIN(nn.Module):
    def __init__(self, style_dim, num_features):
        super().__init__()
        self.num_features = num_features
        self.style_mlp = nn.Sequential(
            nn.Linear(style_dim, 128), nn.ReLU(),
            nn.Linear(128, num_features * 2)
        )

    def forward(self, x, z_t):
        style_params = self.style_mlp(z_t)
        gamma, beta = style_params.chunk(2, dim=1)
        gamma = gamma.view(-1, self.num_features, 1, 1)
        beta = beta.view(-1, self.num_features, 1, 1)
        x_norm = F.instance_norm(x)
        return x_norm * (1 + gamma) + beta

In [10]:
class ModulatedGenerator(nn.Module):
    def __init__(self, style_dim=16, base_channels=64):
        super().__init__()
        self.adain1 = AdaIN(style_dim, base_channels)
        self.upconv1 = nn.ConvTranspose2d(base_channels, 32, 4, 2, 1) # to 14x14
        
        self.adain2 = AdaIN(style_dim, 32)
        self.upconv2 = nn.ConvTranspose2d(32, 1, 4, 2, 1) # to 28x28
        
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, blueprint, z_t):
        x = self.adain1(blueprint, z_t)
        x = self.leaky_relu(x)
        x = self.upconv1(x)
        
        x = self.adain2(x, z_t)
        x = self.leaky_relu(x)
        x = self.upconv2(x)
        return torch.sigmoid(x)

In [12]:
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            spectral_norm(nn.Conv2d(1, 32, 4, 2, 1)), nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv2d(32, 64, 4, 2, 1)), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 1, 3, 1, 1), nn.Sigmoid() 
        )

    def forward(self, x):
        return self.model(x)

In [13]:
def train_mst_vae_gan(dataloader, encoder, base_decoder, generator, discriminator, 
                      opt_G, opt_D, device, epochs):
    
    print("Starting MST-VAE-GAN Training...")
    ep_d_cost, ep_l1_cost, ep_kl_cost, ep_g_adv_cost = [], [], [], []
    
    for epoch in range(epochs):
        ep_d_loss, ep_l1_loss, ep_kl_loss, ep_g_adv_loss = 0, 0, 0, 0
        
        for batch, _ in dataloader:
            real_images = batch.to(device)
            batch_size = real_images.size(0)
            
            # Forward Pass
            mu, logvar = encoder(real_images)
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z_s = mu + eps * std
            
            blueprint = base_decoder(z_s)
            z_t = torch.randn(batch_size, 16).to(device)
            fake_images = generator(blueprint, z_t)

            # Train Patch Discriminator
            opt_D.zero_grad()
            real_patch_preds = discriminator(real_images)
            fake_patch_preds = discriminator(fake_images.detach())
            
            valid = torch.ones_like(real_patch_preds) * 0.9 # Label smoothing
            fake = torch.zeros_like(fake_patch_preds)
            
            d_loss_real = F.binary_cross_entropy(real_patch_preds, valid)
            d_loss_fake = F.binary_cross_entropy(fake_patch_preds, fake)
            d_loss = (d_loss_real + d_loss_fake) / 2
            
            d_loss.backward()
            opt_D.step()

            # Train Generator/VAE Stream
            opt_G.zero_grad()
            
            l1_loss = F.l1_loss(fake_images, real_images, reduction='sum') / batch_size
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
            
            fake_patch_preds_for_G = discriminator(fake_images)
            g_adv_loss = F.binary_cross_entropy(fake_patch_preds_for_G, valid)
            
            beta = 1.0    # KL weight
            gamma = 0.1   # GAN weight (keep small so L1 structure isn't lost)
            total_g_loss = l1_loss + (beta * kl_loss) + (gamma * g_adv_loss)
            
            total_g_loss.backward()
            opt_G.step()
            
            # Tracking
            ep_d_loss += d_loss.item()
            ep_l1_loss += l1_loss.item()
            ep_kl_loss += kl_loss.item()
            ep_g_adv_loss += g_adv_loss.item()
            
        # Averages for the epoch
        num_batches = len(dataloader)
        ep_d_cost.append(ep_d_loss/num_batches)
        ep_l1_cost.append(ep_l1_loss/num_batches)
        ep_kl_cost.append(ep_kl_loss/num_batches)
        ep_g_adv_cost.append(ep_g_adv_loss/num_batches)
        print(f"Epoch {epoch+1}/{epochs} | "
              f"D_Loss: {ep_d_loss/num_batches:.3f} | "
              f"L1_Recon: {ep_l1_loss/num_batches:.3f} | "
              f"KL: {ep_kl_loss/num_batches:.3f} | "
              f"G_Adv: {ep_g_adv_loss/num_batches:.3f}")
        
    return ep_d_cost, ep_l1_cost, ep_kl_cost, ep_g_adv_cost

In [14]:
def generate_standalone_images(base_decoder, generator, device, latent_dim=16, style_dim=16, n_images=16):
    """
    Generates completely novel images from pure noise, bypassing the Encoder.
    """
    base_decoder.eval()
    generator.eval()
    
    with torch.no_grad():
        # 1. Sample random structural blueprints (Z_s)
        # Because of the KL loss, random noise maps to valid digit structures!
        z_s = torch.randn(n_images, latent_dim).to(device)
        blueprint = base_decoder(z_s) 
        
        # 2. Sample random textures/styles (Z_t)
        z_t = torch.randn(n_images, style_dim).to(device)
        
        # 3. Generate final images
        generated = generator(blueprint, z_t).cpu()

    # Plot the entirely generated images in a 4x4 grid
    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(generated[i].squeeze(), cmap="gray")
        ax.axis("off")

    plt.suptitle("Standalone Generations (Pure Noise to Image)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Return models to training mode
    base_decoder.train()
    generator.train()

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize Models
LATENT_DIM = 16
STYLE_DIM = 16

encoder = StructureEncoder(latent_dim=LATENT_DIM).to(device)
base_decoder = BaseDecoder(latent_dim=LATENT_DIM, base_channels=64).to(device)
generator = ModulatedGenerator(style_dim=STYLE_DIM, base_channels=64).to(device)
discriminator = PatchDiscriminator().to(device)

lr = 2e-4
opt_G = optim.Adam(
    list(encoder.parameters()) + list(base_decoder.parameters()) + list(generator.parameters()),
    lr=lr, betas=(0.5, 0.999)
)
opt_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Train
ep_d_loss, ep_l1_cost, ep_kl_loss, ep_g_adv_cost = train_mst_vae_gan(
    dataloader=train_dataloader,
    encoder=encoder,
    base_decoder=base_decoder,
    generator=generator,
    discriminator=discriminator,
    opt_G=opt_G,
    opt_D=opt_D,
    device=device,
    epochs=1)

Using device: mps
Starting MST-VAE-GAN Training...
Epoch 1/1 | D_Loss: 0.613 | L1_Recon: 98.641 | KL: 10.041 | G_Adv: 0.891
